Calculate distances between all pairs of species

In [22]:
from Bio import Phylo
import numpy as np

# from scipy.spatial.distance import pdist, squareform

# Load phylogenetic tree
tree = Phylo.read(
    r"..\..\input\aligned_caffeine_without_nodes_w_no_coords_tree.nwk", "newick"
)


# Extract terminal node names (species)
terminals = tree.get_terminals()
n = len(terminals)

# Initialize an empty distance matrix
phylo_distances = np.zeros((n, n))

# Compute pairwise distances
for i, sp1 in enumerate(terminals):
    for j, sp2 in enumerate(terminals):
        phylo_distances[i, j] = tree.distance(sp1, sp2)

# Optionally, print or convert the matrix to a DataFrame for better readability
import pandas as pd

species_names = [str(sp) for sp in terminals]
phylo_df = pd.DataFrame(phylo_distances, index=species_names, columns=species_names)

phylo_df.to_csv("../data/gen_dist_matrix.csv")
print(phylo_df)

                         C_kianjavatensis_A602  C_bertrandii_A5  \
C_kianjavatensis_A602                  0.00000          0.13751   
C_bertrandii_A5                        0.13751          0.00000   
C_richardii_A575                       0.12301          0.11224   
C_farafanganensis_A208                 0.13209          0.12132   
C_millotii_A222                        0.12205          0.11128   
C_abbayesii_A601                       0.11616          0.10539   
C_homollei_A945                        0.11978          0.11299   
C_perrieri_A12                         0.13163          0.12484   
C_leroyi_A315                          0.11986          0.11307   
C_andrambovatensis_A310                0.13703          0.13024   
C_vianneyi_A946                        0.12842          0.12301   
C_resinosa_A8                          0.13620          0.13079   
C_arenesiana_A403                      0.14402          0.13861   
C_vatovavyensis_A830                   0.14993          0.1445

Calculate geographic distances between species locations

In [2]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic  # For Haversine-based distances

# Load CSV file
geo_df = pd.read_csv(r"..\data\dr_guyot_all_for_collection.csv")
print(geo_df.head())

        specimen_id   latitude  longitude
0  Coffea_abbayesii -24.366660  46.450000
1  Coffea_abbayesii -24.730000  46.830000
2  Coffea_abbayesii -24.733300  46.833300
3  Coffea_abbayesii -24.666670  46.833330
4  Coffea_abbayesii -24.733333  46.833333


In [3]:

genetic_df = phylo_df.copy()
# Extract tree node names
tree_node_names = genetic_df.index.tolist()

# Step 1: Create mapping from "Coffea_abbayesii" → "C_abbayesii_A601"
mapping = {}
for node in tree_node_names:
    parts = node.split("_")
    if len(parts) >= 3:
        species_name = f"Coffea_{parts[1]}"
        if species_name not in mapping:
            mapping[species_name] = node

# Step 2: Apply mapping to geo_df
geo_df["mapped_name"] = geo_df["specimen_id"].map(mapping)

# Step 3: Drop unmatched and finalize
geo_df_cleaned = geo_df.dropna(subset=["mapped_name"]).copy()
geo_df_cleaned = geo_df_cleaned[["mapped_name", "latitude", "longitude"]]
geo_df_cleaned = geo_df_cleaned.rename(columns={"mapped_name": "specimen_id"})

# Step 4: Save cleaned file
output_path = "../data/dr_guyot_collection_specim_renamed.csv"
geo_df_cleaned.to_csv(output_path, index=False)

geo_df_cleaned.head()


,specimen_id,latitude,longitude
0,C_abbayesii_A601,-24.366660,46.450000
1,C_abbayesii_A601,-24.730000,46.830000
2,C_abbayesii_A601,-24.733300,46.833300
3,C_abbayesii_A601,-24.666670,46.833330
4,C_abbayesii_A601,-24.733333,46.833333


Only one  geolocation point per species is necessary to calculate mantel test. Exercise to be done with median, mean, hand-picked


In [6]:
# Group by species and calculate median coordinates
median_coords = (
    geo_df_cleaned.groupby("specimen_id")[["longitude", "latitude"]].median().reset_index()
)
print(median_coords)

                specimen_id  longitude   latitude
0          C_abbayesii_A601  46.962222 -24.733333
1   C_andrambovatensis_A310  47.758333 -21.516667
2         C_arenesiana_A403  48.216667 -18.950000
3           C_bertrandii_A5  46.674722 -25.033330
4           C_dubardii_A969  49.106944 -12.910000
5    C_farafanganensis_A208  47.863889 -21.383333
6             C_heimii_A516  49.347115 -12.396327
7           C_homollei_A945  48.731111 -17.916667
8        C_humbertii_RNF785  44.283333 -23.216667
9     C_kianjavatensis_A602  47.862639 -21.383333
10        C_lancifolia_A320  49.203333 -17.931667
11            C_leroyi_A315  48.757219 -17.495553
12          C_millotii_A222  49.326667 -17.533330
13           C_perrieri_A12  45.989986 -21.382500
14       C_pervilleana_A957  45.920397 -17.543610
15            C_resinosa_A8  49.133333 -18.791944
16         C_richardii_A575  49.405417 -17.666656
17         C_tetragona_A252  44.516667 -18.033610
18        C_tsirananae_A515  49.297935 -12.323889


In [7]:
# Group by species and calculate mean coordinates
mean_coords = (
    geo_df_cleaned.groupby("specimen_id")[["longitude", "latitude"]].mean().reset_index()
)
print(mean_coords)

                specimen_id  longitude   latitude
0          C_abbayesii_A601  47.108862 -24.072093
1   C_andrambovatensis_A310  47.791667 -21.475000
2         C_arenesiana_A403  48.067106 -19.023884
3           C_bertrandii_A5  46.788074 -24.631393
4           C_dubardii_A969  48.683701 -13.849855
5    C_farafanganensis_A208  47.787239 -22.057978
6             C_heimii_A516  49.133383 -13.500596
7           C_homollei_A945  48.745709 -18.501110
8        C_humbertii_RNF785  44.276980 -23.121770
9     C_kianjavatensis_A602  47.878680 -21.383820
10        C_lancifolia_A320  48.757778 -19.082222
11            C_leroyi_A315  48.441704 -17.711347
12          C_millotii_A222  49.061602 -17.700635
13           C_perrieri_A12  45.757698 -20.744695
14       C_pervilleana_A957  46.252573 -16.601791
15            C_resinosa_A8  48.802757 -19.178310
16         C_richardii_A575  49.412444 -17.214852
17         C_tetragona_A252  46.141376 -17.126044
18        C_tsirananae_A515  49.242499 -12.595645


KeyError: "['C_dolichophylla_A206', 'C_ambodirianensis_A572', 'C_andrambovatensis_A310', 'C_vatovavyensis_A830', 'C_mauritiana_Makes4', 'C_mauritiana_BM17_25'] not in index"

In [8]:
from sklearn.cluster import DBSCAN


# Only keep necessary columns
coords_df = geo_df_cleaned[["specimen_id", "latitude", "longitude"]].dropna()


# Function to compute centroid of largest cluster per species
def compute_cluster_centroids(df):
    centroids = []
    for name, group in df.groupby("specimen_id"):
        coords = group[["latitude", "longitude"]].values

        if len(coords) < 3:
            # If fewer than 3 points, skip clustering and just take mean
            centroid = coords.mean(axis=0)
        else:
            # Apply DBSCAN to find clusters
            db = DBSCAN(eps=0.5, min_samples=2).fit(coords)
            labels = db.labels_
            if np.all(labels == -1):
                # All outliers, fall back to mean
                centroid = coords.mean(axis=0)
            else:
                # Find largest cluster
                largest_label = pd.Series(labels).value_counts().idxmax()
                cluster_points = coords[labels == largest_label]
                centroid = cluster_points.mean(axis=0)

        centroids.append(
            {"specimen_id": name, "latitude": centroid[0], "longitude": centroid[1]}
        )

    return pd.DataFrame(centroids)


# Compute cluster-based centroids
centroid_coords_df = compute_cluster_centroids(coords_df)

In [9]:
centroid_coords_df

,specimen_id,latitude,longitude
0,C_abbayesii_A601,-24.744807,46.919542
1,C_andrambovatensis_A310,-21.475000,47.791667
2,C_arenesiana_A403,-19.023884,48.067106
3,C_bertrandii_A5,-25.037400,46.653250
4,C_dubardii_A969,-12.759268,49.141778
5,C_farafanganensis_A208,-21.382741,47.864843
6,C_heimii_A516,-12.418727,49.381558
7,C_homollei_A945,-21.382917,47.865278
8,C_humbertii_RNF785,-23.420697,44.101995
9,C_kianjavatensis_A602,-21.383820,47.878680


In [18]:
import pandas as pd
from geopy.distance import geodesic

# Initialize an empty DataFrame for pairwise distances
species = centroid_coords_df['specimen_id'].tolist()
geo_distance_df = pd.DataFrame(index=species, columns=species, dtype=float)

# Compute pairwise geographic distances
for i in species:
    lat_lon_i = centroid_coords_df[centroid_coords_df['specimen_id'] == i][['latitude', 'longitude']].values[0]
    for j in species:
        if pd.isna(geo_distance_df.loc[i, j]):
            lat_lon_j = centroid_coords_df[centroid_coords_df['specimen_id'] == j][['latitude', 'longitude']].values[0]
            dist = geodesic(lat_lon_i, lat_lon_j).kilometers
            geo_distance_df.loc[i, j] = dist
            geo_distance_df.loc[j, i] = dist

# Save to file
geo_distance_df.to_csv("../data/geo_dist_matrix_centroid.csv")

Performing Mantel test

In [23]:
from skbio.stats.distance import mantel, DistanceMatrix
import pandas as pd

# Load data
genetic_df = pd.read_csv("../data/gen_dist_matrix.csv", index_col=0)
geo_df = pd.read_csv("../data/geo_dist_matrix_centroid.csv", index_col=0)
metadata_df = pd.read_csv("../data/data_w_series_habitat.csv")

In [36]:
shared_species = sorted(
    set(genetic_df.index) & set(geo_df.index) & set(metadata_df["Species_name"])
)
genetic_df = genetic_df.loc[shared_species, shared_species]
geo_df = geo_df.loc[shared_species, shared_species]
metadata_df = metadata_df[metadata_df["Species_name"].isin(shared_species)]
metadata_df[["Species_name", "Botanical series"]].to_csv("../data/matched_specimen_metadata.csv", index=False)

In [37]:
metadata_df

,Species_name,caffeine_percent,Population code,Species,Botanical series,Origin/locality,Province,Latitude,Longitude,Habitat class
0,C_tetragona_A252,0.030,A252,C. tetragona Jum. & H.Perrier,Garcinioides,Behangony (Est de Maromandia),Mahajanga (Northwest),14°13′30″ S,48°09′ E,3.0
1,C_dubardii_A969,0.000,A969,C. dubardii Jum.,Garcinioides,North of Vohemar (Diégo-Suarez),Antsiranana (North),13°20′ S,49°57′ E,3.0
2,C_heimii_A516,0.000,A516,C. heimii J.-F.Leroy,Garcinioides,Diégo-Suarez (Sahafary),Antsiranana (North),12°35′ S,49°26′30″ E,3.0
3,C_farafanganensis_A208,0.045,A208,C. farafanganensis J.-F.Leroy,Millotii complex,Farafangana (Amboangibe),Fianarantsoa (Southeast),22°53′ S,47°48′ E,1.0
6,C_richardii_A575,0.030,A575,C. richardii J.-F.Leroy,Millotii complex,Fenerive-Est (Tampolo),Toamasina (East),17°17′ S,49°25′ E,1.0
7,C_abbayesii_A601,0.000,A601,C. abbayesii J.-F.Leroy,Millotii complex,Fort-Dauphin (Isaka-Ivondro),Toliara (Southeast),24°45′15″ S,46°51′45″ E,2.0
8,C_millotii_A222,0.000,A222,C. millotii J.-F.Leroy,Millotii complex,Mananjary (Tolongoina),Fianarantsoa (Southeast),21°31′ S,47°25′ E,2.0
9,C_leroyi_A315,0.020,A315,C. leroyi A.P.Davis,Multiflorae,Ifanadiana (Ambodiarafia),Fianarantsoa (Southeast),21°20′ S,47°45′ E,1.0
10,C_andrambovatensis_A310,0.000,A310,C. andrambovatensis J.-F.Leroy,Multiflorae,Vatovavy,Fianarantsoa (Southeast),21°24′3″ S,47°56′32″ E,1.0
11,C_resinosa_A8,0.000,A8,C. resinosa (Hook.f.) Radlk.,Multiflorae,Nosy Varika,Fianarantsoa (Southeast),20°36′26.6″ S,48°31′56.5″ E,1.0


In [38]:
print("\nSpecies count per botanical series (after alignment):")
for series, group in metadata_df.groupby("Botanical series"):
    print(f"{series}: {len(group)} species")

# Run Mantel tests by series with at least 3 species
results = []
for series, group in metadata_df.groupby("Botanical series"):
    species = group["Species_name"].tolist()
    if len(species) < 3:
        continue

    g_dm = DistanceMatrix(genetic_df.loc[species, species].values, ids=species)
    d_dm = DistanceMatrix(geo_df.loc[species, species].values, ids=species)

    r, p, _ = mantel(g_dm, d_dm, method="pearson", permutations=10000)
    results.append(
        {
            "Botanical series": series,
            "n_species": len(species),
            "Mantel_r": round(r, 3),
            "p_value": round(p, 4),
        }
    )

# Output results
mantel_df = pd.DataFrame(results)

if "p_value" in mantel_df.columns:
    print("\nMantel Test Results:")
    print(mantel_df.sort_values("p_value"))
else:
    print("\nNo valid series with ≥3 species found for Mantel testing.")
# Output results
mantel_df = pd.DataFrame(results)
print(mantel_df.sort_values("p_value"))


Species count per botanical series (after alignment):
Garcinioides: 3 species
Millotii complex: 4 species
Multiflorae: 7 species
Subterminales: 3 species
Verae: 3 species

Mantel Test Results:
   Botanical series  n_species  Mantel_r  p_value
2       Multiflorae          7    -0.502   0.0714
1  Millotii complex          4    -0.718   0.0833
0      Garcinioides          3     1.000   0.1668
4             Verae          3     0.980   0.3340
3     Subterminales          3     0.853   0.4989
   Botanical series  n_species  Mantel_r  p_value
2       Multiflorae          7    -0.502   0.0714
1  Millotii complex          4    -0.718   0.0833
0      Garcinioides          3     1.000   0.1668
4             Verae          3     0.980   0.3340
3     Subterminales          3     0.853   0.4989


mantel for each sample

In [56]:
df = pd.read_csv("../data/dr_guyot_collection_specim_renamed.csv")

df["specimen_id"] = (
    df["specimen_id"]
    + "_"
    + (df.groupby("specimen_id").cumcount() + 1).astype(str)
)

df.to_csv("../data/dr_guyot_collection_specim_renamed_increment.csv", index=False)

In [57]:
df

,specimen_id,latitude,longitude
0,C_abbayesii_A601_1,-24.366660,46.450000
1,C_abbayesii_A601_2,-24.730000,46.830000
2,C_abbayesii_A601_3,-24.733300,46.833300
3,C_abbayesii_A601_4,-24.666670,46.833330
4,C_abbayesii_A601_5,-24.733333,46.833333
...,...,...,...
449,C_vatovavyensis_A830_1,-21.383333,47.866667
450,C_vatovavyensis_A830_2,-21.400000,47.933333
451,C_vianneyi_A946_1,-21.382389,47.863972
452,C_vianneyi_A946_2,-21.383333,47.866667


In [60]:
from sklearn.metrics.pairwise import haversine_distances

coords_rad = np.radians(df[["latitude","longitude"]].to_numpy())
dist_matrix = haversine_distances(coords_rad, coords_rad) * 6371.0  # Earth radius in km

# 3. Wrap it in a DataFrame and save
dist_df = pd.DataFrame(
    dist_matrix,
    index=df["specimen_id"],
    columns=df["specimen_id"]
)
dist_df.to_csv("../data/geographic_dist_matrix_full_specimens_optimized.csv")

In [ ]:
geo_df = pd.read_csv(
    "../data/geographic_dist_matrix_full_specimens_optimized.csv", index_col=0
)

# Reuse these specimen names to replicate the genetic distances accordingly
# We'll use the tree-level distances and duplicate them for each specimen mapping

# Reload tree and extract distances between tip nodes
from Bio import Phylo
from io import StringIO

with open("../../input/aligned_caffeine_without_nodes_w_no_coords_tree.nwk", "r") as f:
    tree_data = f.read()
tree = Phylo.read(StringIO(tree_data), "newick")

# Extract tip distances
from collections import defaultdict
from Bio.Phylo.TreeConstruction import _Matrix


def get_tree_distances(tree):
    terminals = tree.get_terminals()
    distances = defaultdict(dict)
    for i, t1 in enumerate(terminals):
        for j, t2 in enumerate(terminals):
            d = tree.distance(t1, t2)
            distances[t1.name][t2.name] = d
    return distances


tip_distances = get_tree_distances(tree)

# Build a mapping from specimen name to its tree node
specimen_to_tip = {name: "_".join(name.split("_")[:3]) for name in geo_df.index}
unique_specimens = geo_df.index.tolist()

# Build specimen-level genetic distance matrix
genetic_distances = pd.DataFrame(
    index=unique_specimens, columns=unique_specimens, dtype=float
)

for i in unique_specimens:
    for j in unique_specimens:
        tip_i = specimen_to_tip[i]
        tip_j = specimen_to_tip[j]
        genetic_distances.loc[i, j] = tip_distances[tip_i][tip_j]

# Save final specimen-level genetic distance matrix
genetic_distances.to_csv("../data/genetic_dist_matrix_full_specimens.csv")


,C_abbayesii_A601_1,C_abbayesii_A601_2,C_abbayesii_A601_3,C_abbayesii_A601_4,C_abbayesii_A601_5
C_abbayesii_A601_1,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_2,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_3,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_4,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_5,0.0,0.0,0.0,0.0,0.0


In [62]:
import pandas as pd
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import mantel

# 2) Load your two CSVs (rows and columns in the same order; first column is the ID)
geo_df = pd.read_csv("../data/geographic_dist_matrix_full_specimens_optimized.csv",  index_col=0)
gen_df = pd.read_csv("../data/genetic_dist_matrix_full_specimens.csv",      index_col=0)

# 3) Ensure the order of samples matches exactly in both
ids = geo_df.index.intersection(gen_df.index).tolist()
geo_df = geo_df.loc[ids, ids]
gen_df = gen_df.loc[ids, ids]

# 4) Wrap them as scikit-bio DistanceMatrix objects
geo_dm = DistanceMatrix(geo_df.values, ids)
gen_dm = DistanceMatrix(gen_df.values, ids)

# 5) Run the Mantel test
mantel_stat, p_value, n_perms = mantel(
    geo_dm,
    gen_dm,
    method='pearson',     # or 'spearman'
    permutations=10000       # number of random permutations
)

print(f"Mantel r = {mantel_stat:.4f}, p = {p_value:.4f} (n = {n_perms})")

Mantel r = 0.2566, p = 0.0001 (n = 454)


Individual coords grouped by botanical series

In [64]:
import pandas as pd
import re
from skbio.stats.distance import DistanceMatrix, mantel

# Load data
genetic_df = pd.read_csv("../data/genetic_dist_matrix_full_specimens.csv", index_col=0)
geo_df = pd.read_csv("../data/geographic_dist_matrix_full_specimens_optimized.csv", index_col=0)
metadata_raw = pd.read_csv("../data/matched_specimen_metadata.csv")

# Extract base species name (remove specimen increment suffixes like _1, _2)
def extract_base_name(label):
    match = re.match(r"^(C_[a-z]+_[A-Z0-9]+)", label)
    return match.group(1) if match else label

# Create a mapping from specimen_id to base name and merge with metadata
specimen_ids = genetic_df.index.tolist()
base_name_map = {specimen: extract_base_name(specimen) for specimen in specimen_ids}

# Create metadata with specimen-level granularity
metadata = pd.DataFrame({
    "specimen_id": list(base_name_map.keys()),
    "Species_name": list(base_name_map.values())
})
metadata = metadata.merge(metadata_raw, on="Species_name", how="left")

# Filter common specimens
common_specimens = sorted(set(genetic_df.index) & set(geo_df.index) & set(metadata["specimen_id"]))
genetic_df = genetic_df.loc[common_specimens, common_specimens]
geo_df = geo_df.loc[common_specimens, common_specimens]
metadata = metadata[metadata["specimen_id"].isin(common_specimens)]

# Mantel test per botanical series
results = []
for series, group in metadata.groupby("Botanical series"):
    specimens = group["specimen_id"].tolist()
    if len(specimens) < 3:
        continue
    g_dm = DistanceMatrix(genetic_df.loc[specimens, specimens].values, ids=specimens)
    d_dm = DistanceMatrix(geo_df.loc[specimens, specimens].values, ids=specimens)
    r, p, _ = mantel(g_dm, d_dm, method="pearson", permutations=10000)
    results.append({
        "Botanical series": series,
        "n_specimens": len(specimens),
        "Mantel_r": round(r, 4),
        "p_value": round(p, 4)
    })

mantel_df = pd.DataFrame(results).sort_values("p_value")
mantel_df


,Botanical series,n_specimens,Mantel_r,p_value
0,Garcinioides,79,0.3774,0.0001
1,Millotii complex,100,0.2996,0.0001
2,Multiflorae,150,0.1637,0.0001
3,Subterminales,72,0.5397,0.0001
4,Verae,16,0.0887,0.2243
